# Estadísticas descriptivas de parámetros de laboratorio

Este notebook realiza un análisis exploratorio de los parámetros de calidad de agua obtenidos en laboratorio.

El propósito es revisar la distribución, variabilidad, valores faltantes, posibles valores extremos y relaciones entre parámetros antes o después del flujo principal de modelado.

Este notebook es complementario y no hace parte obligatoria del pipeline de entrenamiento. Su función es apoyar la interpretación inicial de los datos de laboratorio.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

In [ ]:
BASE_DIR = Path.cwd()

while not (BASE_DIR / "src").exists() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR))

print("BASE_DIR detectado:")
print(BASE_DIR)

## Configuración de rutas

Se define el archivo de entrada que contiene los parámetros de laboratorio.

El archivo puede ser:

- GeoPackage (`.gpkg`);
- shapefile (`.shp`);
- Excel (`.xlsx`);
- CSV (`.csv`).

Por defecto se usa el archivo procesado del flujo de calidad de agua, que integra puntos, reflectancias, índices espectrales y parámetros de laboratorio.


In [ ]:
# ============================================================
# RUTAS
# ============================================================

ruta_datos = (
    BASE_DIR
    / "data"
    / "processed"
    / "calidad_agua"
    / "Puntos_Muestreo_Reflectancia_Indices_Excel_2.gpkg"
)

carpeta_tablas = (
    BASE_DIR
    / "outputs"
    / "tables"
    / "calidad_agua"
    / "estadisticas_laboratorio"
)

carpeta_figuras = (
    BASE_DIR
    / "outputs"
    / "figures"
    / "calidad_agua"
    / "estadisticas_laboratorio"
)

carpeta_tablas.mkdir(parents=True, exist_ok=True)
carpeta_figuras.mkdir(parents=True, exist_ok=True)

print("Archivo de entrada:")
print(ruta_datos)

print("\nCarpeta de tablas:")
print(carpeta_tablas)

print("\nCarpeta de figuras:")
print(carpeta_figuras)

## Funciones auxiliares

Se definen funciones simples para cargar datos, validar columnas y generar estadísticas descriptivas.


In [ ]:
def cargar_datos_laboratorio(ruta: Path) -> pd.DataFrame:
    """
    Carga datos tabulares o espaciales con parámetros de laboratorio.
    """
    ruta = Path(ruta)

    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")

    suffix = ruta.suffix.lower()

    if suffix in [".gpkg", ".shp", ".geojson"]:
        gdf = gpd.read_file(ruta)
        return pd.DataFrame(gdf.drop(columns="geometry", errors="ignore"))

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(ruta)

    if suffix == ".csv":
        return pd.read_csv(ruta)

    raise ValueError(
        "Formato no soportado. Use .gpkg, .shp, .geojson, .xlsx, .xls o .csv"
    )


def validar_columnas(df: pd.DataFrame, columnas: list[str]) -> list[str]:
    """
    Retorna las columnas de laboratorio disponibles en el DataFrame.
    """
    disponibles = [col for col in columnas if col in df.columns]
    faltantes = [col for col in columnas if col not in df.columns]

    print("Columnas disponibles:")
    print(disponibles)

    if faltantes:
        print("\nColumnas no encontradas:")
        print(faltantes)

    if not disponibles:
        raise ValueError("No se encontró ninguna columna de laboratorio configurada.")

    return disponibles


def resumen_estadistico(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    """
    Calcula estadísticas descriptivas ampliadas para columnas numéricas.
    """
    data = df[columnas].copy()

    for col in columnas:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    resumen = data.describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]).T

    resumen["missing"] = data.isna().sum()
    resumen["missing_pct"] = data.isna().mean() * 100
    resumen["cv_pct"] = (resumen["std"] / resumen["mean"].replace(0, np.nan)) * 100

    resumen = resumen.rename(columns={
        "count": "n",
        "mean": "media",
        "std": "desv_est",
        "min": "min",
        "5%": "p05",
        "25%": "p25",
        "50%": "mediana",
        "75%": "p75",
        "95%": "p95",
        "max": "max"
    })

    return resumen


def guardar_figura(ruta: Path, dpi: int = 300) -> None:
    """
    Guarda la figura activa.
    """
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(ruta, dpi=dpi, bbox_inches="tight")
    print("Figura guardada en:")
    print(ruta)


## Carga de datos

Se carga el archivo de entrada y se revisan sus dimensiones y columnas disponibles.


In [ ]:
df = cargar_datos_laboratorio(ruta_datos)

print("Dimensiones del DataFrame:")
print(df.shape)

print("\nColumnas:")
print(df.columns.tolist())

display(df.head())

## Selección de parámetros de laboratorio

Se define la lista de parámetros de calidad de agua que se analizarán.

La lista puede ajustarse según las columnas disponibles en el archivo de entrada.


In [ ]:
parametros_laboratorio = [
    "DQO",
    "pH",
    "Fosfatos",
    "CE",
    "Turbidez",
    "Nitratos",
    "Sulfatos",
    "ficocianin",
    "Chl"
]

parametros_disponibles = validar_columnas(
    df=df,
    columnas=parametros_laboratorio
)

df_lab = df[parametros_disponibles].copy()

for col in parametros_disponibles:
    df_lab[col] = pd.to_numeric(df_lab[col], errors="coerce")

display(df_lab.head())

## Estadísticas descriptivas

Se calculan estadísticas descriptivas para cada parámetro:

- número de datos;
- media;
- desviación estándar;
- mínimo;
- percentiles;
- mediana;
- máximo;
- datos faltantes;
- coeficiente de variación.


In [ ]:
tabla_resumen = resumen_estadistico(
    df=df,
    columnas=parametros_disponibles
)

ruta_resumen = carpeta_tablas / "resumen_estadistico_laboratorio.csv"

tabla_resumen.to_csv(
    ruta_resumen,
    encoding="utf-8-sig"
)

print("Resumen guardado en:")
print(ruta_resumen)

display(tabla_resumen)

## Valores faltantes

Se revisa la cantidad y porcentaje de datos faltantes por parámetro.


In [ ]:
tabla_faltantes = (
    pd.DataFrame({
        "parametro": parametros_disponibles,
        "faltantes": df_lab[parametros_disponibles].isna().sum().values,
        "faltantes_pct": df_lab[parametros_disponibles].isna().mean().values * 100
    })
    .sort_values("faltantes_pct", ascending=False)
)

ruta_faltantes = carpeta_tablas / "valores_faltantes_laboratorio.csv"

tabla_faltantes.to_csv(
    ruta_faltantes,
    index=False,
    encoding="utf-8-sig"
)

display(tabla_faltantes)

## Distribución de parámetros

Se generan histogramas para revisar la distribución de cada parámetro de laboratorio.


In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(parametros_disponibles) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(5 * n_cols, 4 * n_rows)
)

axes = np.array(axes).reshape(-1)

for i, col in enumerate(parametros_disponibles):
    ax = axes[i]

    sns.histplot(
        data=df_lab,
        x=col,
        kde=True,
        ax=ax
    )

    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("Frecuencia")

for j in range(len(parametros_disponibles), len(axes)):
    axes[j].axis("off")

plt.tight_layout()

guardar_figura(carpeta_figuras / "histogramas_parametros_laboratorio.png")
plt.show()

## Diagramas de caja

Se generan diagramas de caja para revisar dispersión y posibles valores extremos.

Para facilitar la comparación visual entre parámetros con escalas diferentes, se usa una versión estandarizada de los datos.


In [ ]:
df_std = df_lab.copy()

for col in parametros_disponibles:
    media = df_std[col].mean()
    desv = df_std[col].std()

    if pd.notnull(desv) and desv != 0:
        df_std[col] = (df_std[col] - media) / desv

df_std_melt = df_std.melt(
    var_name="parametro",
    value_name="valor_estandarizado"
)

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df_std_melt,
    x="parametro",
    y="valor_estandarizado"
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Parámetro")
plt.ylabel("Valor estandarizado")
plt.title("Diagramas de caja de parámetros de laboratorio")

plt.tight_layout()

guardar_figura(carpeta_figuras / "boxplots_parametros_laboratorio.png")
plt.show()

## Matriz de correlación

Se calcula la correlación entre parámetros de laboratorio.

Esta matriz permite explorar asociaciones entre variables fisicoquímicas antes de realizar modelado.


In [ ]:
corr_lab = df_lab[parametros_disponibles].corr(method="pearson")

ruta_corr = carpeta_tablas / "correlacion_parametros_laboratorio.csv"

corr_lab.to_csv(
    ruta_corr,
    encoding="utf-8-sig"
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_lab,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.3,
    cbar_kws={"label": "Correlación Pearson"}
)

plt.title("Matriz de correlación de parámetros de laboratorio")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

guardar_figura(carpeta_figuras / "correlacion_parametros_laboratorio.png")
plt.show()

display(corr_lab)

## Identificación simple de valores extremos

Se identifican posibles valores extremos usando el criterio del rango intercuartílico.

Un valor se marca como extremo si está por debajo de:

`Q1 - 1.5 × IQR`

o por encima de:

`Q3 + 1.5 × IQR`

Este criterio es exploratorio y no implica eliminar datos automáticamente.


In [ ]:
registros_outliers = []

for col in parametros_disponibles:
    serie = df_lab[col].dropna()

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr

    outliers = df_lab[(df_lab[col] < limite_inf) | (df_lab[col] > limite_sup)]

    registros_outliers.append({
        "parametro": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "limite_inferior": limite_inf,
        "limite_superior": limite_sup,
        "n_outliers": len(outliers),
        "outliers_pct": len(outliers) / len(df_lab) * 100
    })

tabla_outliers = pd.DataFrame(registros_outliers)

ruta_outliers = carpeta_tablas / "resumen_outliers_laboratorio.csv"

tabla_outliers.to_csv(
    ruta_outliers,
    index=False,
    encoding="utf-8-sig"
)

display(tabla_outliers)

## Exportación de dataset de laboratorio

Se guarda una versión tabular limpia con los parámetros de laboratorio convertidos a formato numérico.


In [ ]:
ruta_dataset_lab = carpeta_tablas / "dataset_laboratorio_numerico.csv"

df_lab.to_csv(
    ruta_dataset_lab,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset de laboratorio guardado en:")
print(ruta_dataset_lab)

## Productos generados

Al finalizar este notebook se generan tablas y figuras exploratorias.

Tablas:

- `resumen_estadistico_laboratorio.csv`;
- `valores_faltantes_laboratorio.csv`;
- `correlacion_parametros_laboratorio.csv`;
- `resumen_outliers_laboratorio.csv`;
- `dataset_laboratorio_numerico.csv`.

Figuras:

- `histogramas_parametros_laboratorio.png`;
- `boxplots_parametros_laboratorio.png`;
- `correlacion_parametros_laboratorio.png`.

Las salidas se guardan en:

`outputs/tables/calidad_agua/estadisticas_laboratorio/`

`outputs/figures/calidad_agua/estadisticas_laboratorio/`
